# Day 11 — boto3: Talking to Bedrock

**Phase 1 · Module 2: Bedrock Setup** · ~30 min

Days 1–10 you built the agent's *brains* in Python. Today you learn the **phone line** that lets Python actually call a large language model on AWS: **`boto3`**, Amazon's Python SDK. Every Bedrock call in the rest of Phase 1 goes through the three moves you learn here: **make a client → send a `converse` request → read the reply (and its token bill).**

> **Keyless & offline.** A real `boto3` client needs AWS credentials and a paid account. To keep this notebook runnable anywhere, we build a tiny **`FakeBedrockRuntime`** that mimics boto3's method names and — crucially — the *exact response shape* AWS returns. Swap it for a real `boto3.client("bedrock-runtime")` later and **not one line of your calling code changes.** That is the whole point of learning the shapes.


## 1. What `boto3` actually is

`boto3` is one SDK for *all* of AWS. You never talk to a service directly — you ask boto3 for a **client** for a specific service in a specific **region**, then call methods on it:

```python
import boto3
brt = boto3.client("bedrock-runtime", region_name="eu-west-2")   # London
```

Two services matter for us:

| Service name | What it does |
|---|---|
| `bedrock` | **control plane** — list models, manage access, guardrails |
| `bedrock-runtime` | **data plane** — actually *run* inference (`converse`, `invoke_model`) |

Below is our stand-in. It exposes the same method names as the real `bedrock-runtime` client, so everything after this cell is real, copy-pasteable boto3 code.


In [1]:
import json, random

class ClientError(Exception):
    """Mimics botocore.exceptions.ClientError: carries a structured 'response'."""
    def __init__(self, code, message):
        self.response = {"Error": {"Code": code, "Message": message}}
        super().__init__(f"{code}: {message}")

class FakeBedrockRuntime:
    """Offline stand-in for boto3.client('bedrock-runtime'). Same methods, same shapes."""

    def converse(self, modelId, messages, inferenceConfig=None, **kw):
        # emulate AWS validation: unknown model -> ValidationException
        if "claude" not in modelId and "nova" not in modelId and "titan" not in modelId:
            raise ClientError("ValidationException", f"model {modelId} not enabled")
        user_text = messages[-1]["content"][0]["text"]
        reply = f"[{modelId.split('.')[-1]}] Re: '{user_text[:40]}' -> classified as OPERATIONAL spend."
        in_tok = max(1, len(user_text.split()))
        out_tok = max(1, len(reply.split()))
        return {
            "output": {"message": {"role": "assistant",
                                    "content": [{"text": reply}]}},
            "stopReason": "end_turn",
            "usage": {"inputTokens": in_tok, "outputTokens": out_tok,
                       "totalTokens": in_tok + out_tok},
        }

    def invoke_model(self, modelId, body, **kw):
        payload = json.loads(body)
        prompt = payload.get("messages", [{}])[-1].get("content", "")
        out = {"content": [{"type": "text", "text": f"raw-invoke reply to: {prompt[:40]}"}],
               "usage": {"input_tokens": 12, "output_tokens": 9}}
        # invoke_model returns bytes on a streaming body, just like the real API
        return {"body": FakeStreamingBody(json.dumps(out).encode())}

class FakeStreamingBody:
    def __init__(self, data): self._data = data
    def read(self): return self._data

brt = FakeBedrockRuntime()
print("client ready:", type(brt).__name__)


client ready: FakeBedrockRuntime


## 2. `converse` — the one API you should use

AWS gives every chat model a **single unified request shape** called *Converse*. You send a list of `messages`; each message has a `role` (`user`/`assistant`) and a `content` list of blocks. Text lives in `{"text": ...}` blocks. This is the same idea you'll meet in the main Bedrock notebook — learn it once, reuse it for every model.


In [2]:
MODEL = "eu.anthropic.claude-sonnet-4-5-20250929-v1:0"   # a Bedrock model id

resp = brt.converse(
    modelId=MODEL,
    messages=[
        {"role": "user",
         "content": [{"text": "Categorise this transaction: 'AWS EU invoice 420.50 GBP'"}]}
    ],
    inferenceConfig={"maxTokens": 200, "temperature": 0.0},
)

import pprint; pprint.pprint(resp)


{'output': {'message': {'content': [{'text': '[claude-sonnet-4-5-20250929-v1:0] '
                                             "Re: 'Categorise this "
                                             "transaction: 'AWS EU inv' -> "
                                             'classified as OPERATIONAL '
                                             'spend.'}],
                        'role': 'assistant'}},
 'stopReason': 'end_turn',
 'usage': {'inputTokens': 8, 'outputTokens': 13, 'totalTokens': 21}}


☝ Notice the request is **plain Python dicts and lists** — no SDK-specific objects. `inferenceConfig` holds the knobs you already know (`maxTokens`, `temperature`, `topP`). The response is a nested dict; §3 shows how to reach into it *correctly* — the #1 thing people get wrong.


## 3. Parse the reply — reach the text, don't guess

The model's words are buried at `response["output"]["message"]["content"][0]["text"]`. `content` is a **list of blocks** (a model can return several), so you index `[0]` for the first text block. The `usage` dict is your **token bill** — log it on every call, because in production tokens are money.


In [3]:
def read_text(resp) -> str:
    return resp["output"]["message"]["content"][0]["text"]

text = read_text(resp)
usage = resp["usage"]

print("reply :", text)
print("tokens:", usage["inputTokens"], "in +", usage["outputTokens"], "out =", usage["totalTokens"])
print("why it stopped:", resp["stopReason"])


reply : [claude-sonnet-4-5-20250929-v1:0] Re: 'Categorise this transaction: 'AWS EU inv' -> classified as OPERATIONAL spend.
tokens: 8 in + 13 out = 21
why it stopped: end_turn


☝ One tiny helper (`read_text`) means the rest of your codebase never has to remember that ugly path. When you switch to a real client, this function keeps working unchanged — that's the payoff of coding against the **shape**, not the transport.


## 4. `invoke_model` — the older, model-specific API

Before *Converse*, each model family had its **own** JSON body, and you called `invoke_model` with a hand-built `body` string and read **bytes** back. You'll still see it in older code and for features Converse doesn't cover, so recognise the pattern: **`json.dumps` going in, `body.read()` + `json.loads` coming out.**


In [4]:
body = json.dumps({
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 200,
    "messages": [{"role": "user", "content": "Flag if this looks like fraud: 3 logins, 3 countries, 2 min"}],
})

raw = brt.invoke_model(modelId=MODEL, body=body)
parsed = json.loads(raw["body"].read())          # <-- bytes -> dict
print(parsed["content"][0]["text"])
print("usage:", parsed["usage"])


raw-invoke reply to: Flag if this looks like fraud: 3 logins,
usage: {'input_tokens': 12, 'output_tokens': 9}


☝ Same call, more plumbing: **you** own the request/response schema per model. Rule of thumb — reach for **`converse` by default**; drop to `invoke_model` only when you need a model-specific feature. Both are just methods on the same client.


## 5. Handle `ClientError` — by error **code**, not by string

Real API calls fail: throttling, wrong region, model not enabled, bad input. boto3 raises `botocore.exceptions.ClientError`, and the useful part is the **code** at `err.response["Error"]["Code"]`. Branch on that code — never on the human message text (it changes).

```python
from botocore.exceptions import ClientError   # the real import
```


In [5]:
def safe_converse(client, model, prompt):
    try:
        r = client.converse(
            modelId=model,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
        )
        return read_text(r)
    except ClientError as err:
        code = err.response["Error"]["Code"]
        if code == "ThrottlingException":
            return "[retry later] rate limit hit"
        if code == "AccessDeniedException":
            return "[fix IAM] no permission for this model"
        if code == "ValidationException":
            return "[fix request] model not enabled or bad input"
        raise                                   # unknown -> don't swallow

print(safe_converse(brt, MODEL, "hello"))                      # works
print(safe_converse(brt, "cohere.command", "hello"))           # -> ValidationException branch


[claude-sonnet-4-5-20250929-v1:0] Re: 'hello' -> classified as OPERATIONAL spend.
[fix request] model not enabled or bad input


☝ The good model returns text; the un-enabled one is caught and turned into an **actionable** message instead of a stack trace. Branching on the **code** is what makes this robust: AWS may reword the message, but `ThrottlingException` stays `ThrottlingException`. The `raise` at the end is deliberate — an error you didn't plan for should **not** be silently hidden.


## 6. Retry throttling with backoff

`ThrottlingException` (HTTP 429) just means *slow down* — the right response is **wait and retry**, with the wait doubling each time (exponential backoff). This is the manual version; in real code you'd also set `boto3.client(..., config=Config(retries={"max_attempts": 5, "mode": "adaptive"}))` and let botocore do it for you.


In [6]:
import time

class Flaky:
    """Throttles the first two calls, then succeeds — to show backoff."""
    def __init__(self): self.n = 0
    def converse(self, **kw):
        self.n += 1
        if self.n < 3:
            raise ClientError("ThrottlingException", "Too many requests")
        return brt.converse(**kw)

def with_backoff(client, model, prompt, max_attempts=5):
    for attempt in range(max_attempts):
        try:
            return client.converse(
                modelId=model,
                messages=[{"role": "user", "content": [{"text": prompt}]}])
        except ClientError as e:
            if e.response["Error"]["Code"] != "ThrottlingException":
                raise
            wait = 0.05 * (2 ** attempt)          # 50ms, 100ms, 200ms, ...
            print(f"  throttled (try {attempt+1}) -> sleeping {wait:.2f}s")
            time.sleep(wait)
    raise RuntimeError("gave up after retries")

r = with_backoff(Flaky(), MODEL, "categorise: SWIFT fee 15 GBP")
print("OK ->", read_text(r))


  throttled (try 1) -> sleeping 0.05s
  throttled (try 2) -> sleeping 0.10s
OK -> [claude-sonnet-4-5-20250929-v1:0] Re: 'categorise: SWIFT fee 15 GBP' -> classified as OPERATIONAL spend.


☝ Two throttles, two growing sleeps, then success — no crash, no lost request. This exact pattern (catch `ThrottlingException` → back off → retry) is what keeps a batch job of 10,000 invoices alive when Bedrock briefly rate-limits you.


## 7. Recap + checklist

| Move | Code | Watch out for |
|---|---|---|
| **make client** | `boto3.client("bedrock-runtime", region_name=...)` | region must have the model enabled |
| **call** | `client.converse(modelId, messages, inferenceConfig)` | `content` is a **list of blocks** |
| **read** | `resp["output"]["message"]["content"][0]["text"]` | wrap it in a helper |
| **bill** | `resp["usage"]["totalTokens"]` | log tokens on every call |
| **errors** | `except ClientError` → branch on `Error.Code` | never branch on the message string |
| **throttle** | catch `ThrottlingException`, backoff + retry | or set botocore `retries` config |

**Your 30 min today**
- [ ] Say why `converse` beats `invoke_model` for new code (unified shape, no per-model JSON).
- [ ] Write `read_text(resp)` from memory.
- [ ] Catch a `ClientError` and branch on `Error.Code`.
- [ ] Explain what a `ThrottlingException` means and the fix (backoff + retry).

➡ **Next:** the main **Bedrock Foundations** notebook uses this exact `converse` call against the model catalogue (Claude, Nova, Titan, Mistral) and works out the **cost** from that `usage` dict.
